# 01 Auto Deliveries and Margin Model

Key question: how much of Tesla's current valuation can be supported by
auto deliveries, ASP, manufacturing margins and operating leverage before
giving credit to autonomy or robotics?

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from valuation import (
    MarketPremiumInputs,
    OptionScenario,
    SegmentValuation,
    SotpInputs,
    build_sensitivity_grid,
    implied_margin_for_enterprise_value,
    implied_revenue_for_enterprise_value,
    market_premium_value,
    probability_weighted_option_value,
    sotp_equity_value,
)

DATA_DIR = next(
    candidate / "apps/notebooks/studies/tesla_valuation/data"
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "apps/notebooks/studies/tesla_valuation/data").exists()
)
pd.options.display.float_format = "{:,.2f}".format

In [ ]:
assumptions = pd.read_csv(DATA_DIR / "segment_assumptions.csv")
auto = assumptions[assumptions["segment"].eq("Auto")]
auto.pivot_table(index="metric", columns="case", values="value", aggfunc="first")

In [ ]:
cases = ["bear", "base", "bull"]
auto_case = (
    auto.pivot_table(index="case", columns="metric", values="value", aggfunc="first")
    .reindex(cases)
    .assign(
        revenue_usd_bn=lambda df: df["2030 deliveries"] * df["ASP"] / 1_000,
        ebitda_usd_bn=lambda df: df["revenue_usd_bn"] * df["EBITDA margin"],
        fcf_margin=lambda df: df["EBITDA margin"] - [0.05, 0.04, 0.03],
        ev_revenue_multiple=[1.5, 2.5, 4.0],
        ev_ebitda_multiple=[10.0, 15.0, 22.0],
    )
)
auto_case["fcf_usd_bn"] = auto_case["revenue_usd_bn"] * auto_case["fcf_margin"]
auto_case["ev_revenue_value_usd_bn"] = (
    auto_case["revenue_usd_bn"] * auto_case["ev_revenue_multiple"]
)
auto_case["ev_ebitda_value_usd_bn"] = auto_case["ebitda_usd_bn"] * auto_case["ev_ebitda_multiple"]
auto_case["selected_value_usd_bn"] = auto_case[
    ["ev_revenue_value_usd_bn", "ev_ebitda_value_usd_bn"]
].mean(axis=1)
auto_case

In [ ]:
deliveries_asp_grid = build_sensitivity_grid(
    row_values=[4, 5, 6, 7, 8, 10],
    column_values=[32_000, 36_000, 40_000, 44_000],
    formula=lambda deliveries_m, asp: deliveries_m * asp / 1_000,
    row_name="2030 deliveries (m)",
    column_name="ASP ($)",
)
deliveries_asp_grid

In [ ]:
margin_multiple_grid = build_sensitivity_grid(
    row_values=[0.10, 0.14, 0.18, 0.22, 0.26],
    column_values=[1.5, 2.5, 3.5, 4.5],
    formula=lambda margin, multiple: auto_case.loc["base", "revenue_usd_bn"] * margin * multiple,
    row_name="Auto EBITDA margin",
    column_name="EV / revenue",
)
margin_multiple_grid

In [ ]:
ax = auto_case[["revenue_usd_bn", "ebitda_usd_bn", "fcf_usd_bn"]].plot(
    kind="bar", figsize=(9, 4), title="Auto 2030 Case Outputs"
)
ax.set_ylabel("USD bn")
ax.set_xlabel("")
plt.tight_layout()